# ESM2-3B frozen linear-probe extraction: nanobody CDR-H3 kinked/extended

Extracts frozen ESM2-3B (esm2_t36_3B_UR50D) per-layer representations for every nanobody
with a resolved kinked/extended CDR-H3 label, so the probe/control-task analysis (logistic
regression, GroupKFold by framework cluster, shuffled-label control) can run locally on CPU.

No SimpleFold, no folding, no flow sampling -- one forward pass per sequence. Single T4 is
enough; the second bundled GPU is unused.

Saves per target, per ESM2 layer (0..36):
- CDR3 mean-pooled embedding (IMGT 105-117 span)
- the two anchor-residue embeddings (span start, span end-1)

This decides controller-vs-side-stream for the SimpleFold nanobody work: if a frozen linear
probe on these embeddings clears chance + the shuffled-label control, the conformational
signal is already in ESM2 and unused (controller). If not, escalate to a nonlinear probe on
the same saved embeddings before concluding a side-stream (antibody-specific LM) is needed.


In [ ]:
import os, sys, json, time, glob
import numpy as np

WORK = '/kaggle/working'
T0 = time.time()
R = {'note': 'ESM2-3B frozen embedding extraction for kinked/extended CDR-H3 probe', 'rows_done': 0}

def note(t):
    R['stage'] = t
    R['elapsed_min'] = round((time.time() - T0) / 60, 1)
    json.dump(R, open(f'{WORK}/esm_probe_status.json', 'w'), indent=1, default=str)
    print(t, R['elapsed_min'], 'min', flush=True)

import torch
assert torch.cuda.is_available(), 'no GPU'
cap = torch.cuda.get_device_capability(0)
assert cap >= (7, 0), f'sm_{cap[0]}{cap[1]} too old, need sm_70+'
device = 'cuda:0'
print(torch.cuda.get_device_name(0), flush=True)

_ds = glob.glob('/kaggle/input/**/targets_vhh.json', recursive=True)
assert len(_ds) == 1, _ds
TG = {f"{t['pdb']}{t['H']}": t for t in json.load(open(_ds[0]))}
print(len(TG), 'targets total', flush=True)

# Kinked/extended label: same IMGT-115..118 backbone-angle rule already validated on this
# dataset (AUC 0.909 against the independent compactness measure). Recomputed here from the
# crystal CA coordinates already present in targets_vhh.json, so this kernel needs no
# derived-label dataset upload.

def ca_from_hca(hca):
    return np.array([[np.nan] * 3 if r is None or r[1] is None else r[1] for r in hca], float)

def _ang(a, b, c):
    u, v = a - b, c - b
    return float(np.degrees(np.arccos(np.clip(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)), -1, 1))))

def _dih(a, b, c, d):
    b0, b1, b2 = a - b, c - b, d - c
    b1 = b1 / np.linalg.norm(b1)
    v = b0 - np.dot(b0, b1) * b1
    w = b2 - np.dot(b2, b1) * b1
    return float(np.degrees(np.arctan2(np.dot(np.cross(b1, v), w), np.dot(v, w))))

def kink(ca, span):
    hi = span[1]
    idx = [hi - 3, hi - 2, hi - 1, hi]
    if idx[0] < 0 or idx[3] >= len(ca):
        return 'na'
    p = ca[idx]
    if not np.isfinite(p[:, 0]).all():
        return 'na'
    tau = _ang(p[0], p[1], p[2])
    alpha = _dih(p[0], p[1], p[2], p[3])
    if 0 < alpha < 120 and 85 < tau < 130:
        return 'kinked'
    if alpha < -100 and 100 < tau < 145:
        return 'extended'
    return 'other'

LABELED = []
for tid, t in TG.items():
    ca = ca_from_hca(t['hca'])
    lab = kink(ca, tuple(t['cdr3']))
    if lab in ('kinked', 'extended'):
        LABELED.append((tid, t, lab))

print(len(LABELED), 'labeled (kinked/extended) targets', flush=True)
assert len(LABELED) > 400, f'expected ~518, got {len(LABELED)} -- label logic likely wrong'
note('labels ready')


In [ ]:
import subprocess

def must(c, w):
    p = subprocess.run(c, shell=True, text=True, capture_output=True)
    if p.returncode:
        raise RuntimeError(f'{w} (rc={p.returncode}):\n' + (p.stdout + p.stderr)[-2000:])

must(f'{sys.executable} -m pip install -q fair-esm', 'install esm')

w = subprocess.run(
    [sys.executable, '-c', "import torch; torch.hub.load('facebookresearch/esm:main','esm2_t36_3B_UR50D')"],
    capture_output=True, text=True)
if w.returncode:
    raise RuntimeError('esm weight warm-up failed:\n' + (w.stdout + w.stderr)[-2000:])
note('esm cache warm')

model, alphabet = torch.hub.load('facebookresearch/esm:main', 'esm2_t36_3B_UR50D')
model = model.half().to(device).eval()
batch_converter = alphabet.get_batch_converter()

# esm2_t36_3B_UR50D: 36 transformer layers -> representations at levels 0 (embedding) .. 36
NLAYER = 36
LAYERS = list(range(NLAYER + 1))
DIM = 2560

N = len(LABELED)
cdr3_mean = np.zeros((N, len(LAYERS), DIM), dtype=np.float16)
anchor = np.zeros((N, len(LAYERS), 2, DIM), dtype=np.float16)
meta = []

for i, (tid, t, lab) in enumerate(LABELED):
    seq = t['hseq']
    lo, hi = t['cdr3']
    _, _, tokens = batch_converter([(tid, seq)])
    tokens = tokens.to(device)
    with torch.no_grad():
        out = model(tokens, repr_layers=LAYERS, return_contacts=False)
    for li, layer in enumerate(LAYERS):
        h = out['representations'][layer][0, 1:len(seq) + 1].float().cpu().numpy()
        cdr3_mean[i, li] = h[lo:hi].mean(axis=0)
        anchor[i, li, 0] = h[lo]
        anchor[i, li, 1] = h[hi - 1]
    meta.append({'tid': tid, 'kink': lab, 'cdr3_len': hi - lo, 'seq_len': len(seq)})
    if (i + 1) % 50 == 0 or i + 1 == N:
        R['rows_done'] = i + 1
        note(f'{i + 1}/{N} embedded')

np.savez_compressed(f'{WORK}/esm_probe_embeddings.npz', cdr3_mean=cdr3_mean, anchor=anchor)
json.dump(meta, open(f'{WORK}/esm_probe_meta.json', 'w'))
note('done')
print('saved', flush=True)
